# Tool Calling in Agents

This notebook explores different approaches to enabling AI agents to use tools for solving problems. We'll cover:

1. **Baseline LLM** - No tools, relying only on the model's parametric knowledge
2. **ReAct Prompting** - Manual implementation of thought-action-observation loops
3. **LangChain with Local Models** - Using frameworks with local LLMs
4. **LangChain with API Models** - Using frameworks with API-based models
5. **MCP Servers** - Model Context Protocol for tool calling

Each section demonstrates how agents can overcome limitations by using tools for tasks like arithmetic, comparisons, and accessing current information.

## ✅ Completion Checklist

Complete the following TODO sections throughout this notebook:

**Part 2: ReAct Prompting**
- [ ] Implement `compare_numbers()` function
- [ ] Implement `add_numbers()` function
- [ ] Implement `get_current_date()` function
- [ ] Complete `execute_action()` to parse `finish[answer]`
- [ ] Fill in action descriptions in the ReAct prompt
- [ ] Add step to prompt in the ReAct loop

**Part 3: LangChain with Local Models**
- [ ] Implement LangChain tool functions (compare, add, get_date)
- [ ] Complete `parse_tool_call()` function
- [ ] Add tool results to conversation messages

**Part 5: MCP Server**
- [ ] Complete `mcp_server.py` file with all tool implementations

Once you complete all TODO sections, test your implementations by running the cells!

## Setup and Installation

First, let's install all necessary packages for this notebook.

In [ ]:
# Install required packages
!pip install -q transformers torch accelerate langchain langchain-huggingface langchain-anthropic langchain-core python-dotenv rich mcp fastmcp

In [ ]:
# Import common libraries
import random
import re
import json
import torch
import transformers
from datetime import datetime
from rich.console import Console
from rich.panel import Panel

# Initialize console for rich output
console = Console()

# Set seeds for reproducibility
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

transformers.logging.set_verbosity_error()

print("✅ Setup complete!")

## Part 1: Baseline LLM (No Tools)

Let's first see how an LLM performs without any tools. We'll use a small instruction-tuned model and ask it questions that require:
- Mathematical computation
- Number comparison  
- Current date information

These are tasks where LLMs often struggle without external tools.

In [ ]:
# Load a small instruction-tuned model
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, GenerationConfig

# Choose a lightweight model (you can try different ones)
# model_name = "microsoft/Phi-3-mini-4k-instruct"  # 3.8B - Works well
model_name = "Qwen/Qwen2-1.5B-Instruct"  # 1.5B - Smaller, faster but less capable

print(f"Loading model {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    return_full_text=False,
)

generation_config = GenerationConfig(max_new_tokens=200, do_sample=False)


def generate(prompt: str) -> str:
    """Generate text from a prompt."""
    out = gen(prompt, generation_config=generation_config)
    return out[0]["generated_text"]

print("✅ Model loaded!")

In [ ]:
# Test the baseline LLM on various questions
def test_baseline(question: str):
    """Test baseline LLM without tools."""
    prompt = f"Answer this and only this question concisely: {question}"
    console.print(Panel(question, title="Question", border_style="blue"))
    console.print(Panel(prompt, title="Model Input", border_style="cyan"))
    output = generate(prompt)
    console.print(Panel(output, title="Model Output", border_style="green"))
    print()

# Try different questions - notice where the model struggles
test_questions = [
    "9.11 and 9.9, which number is larger?",
    "What is 9.11 + 9.9?",
    "What day is today?",
    "What is 735 / 499?"
]

# Test one question (try changing the index to test different questions)
test_baseline(test_questions[0])

### 🔍 Observations

Without tools, LLMs often:
- Make arithmetic errors (especially with decimal numbers)
- Provide outdated information (can't know today's date)
- Struggle with precise numerical comparisons

**Solution**: Give the agent access to tools!

## Part 2: ReAct Prompting

**ReAct** (Reasoning and Acting) is a prompting method that teaches LLMs to:
1. **Think** about what to do
2. **Act** by calling a tool
3. **Observe** the result
4. **Repeat** until the answer is found

We'll implement a simple ReAct loop manually using carefully designed prompts.

In [ ]:
# Define tools that the agent can use
def compare_numbers(a: float, b: float) -> str:
    """Compare two numbers and return which is greater."""
    # TODO: Implement comparison logic
    # Hint: Use if/elif/else to compare a and b
    # Return a formatted string indicating which number is greater
    # Fill in below:
    pass


def add_numbers(a: float, b: float) -> str:
    """Add two numbers and return the sum."""
    # TODO: Implement addition
    # Calculate a + b and return a formatted string like "a + b is result."
    # Fill in below:
    pass


def get_current_date() -> str:
    """Get the current date."""
    # TODO: Return today's date as a formatted string
    # Hint: Use datetime.now().strftime('%A, %B %d, %Y')
    # Fill in below:
    pass


def execute_action(action: str) -> str:
    """Parse and execute an action, return observation."""
    action = action.strip()
    
    # Parse compare_numbers[a, b]
    match = re.search(r'compare_numbers\[\s*([\d.]+)\s*,\s*([\d.]+)\s*\]', action)
    if match:
        a, b = float(match.group(1)), float(match.group(2))
        return compare_numbers(a, b)
    
    # Parse add_numbers[a, b]
    match = re.search(r'add_numbers\[\s*([\d.]+)\s*,\s*([\d.]+)\s*\]', action)
    if match:
        a, b = float(match.group(1)), float(match.group(2))
        return add_numbers(a, b)
    # TODO: Parse finish[answer] action
    # Hint: Use re.search() to find the pattern 'finish[...]'
    # Extract the answer from inside the brackets
    # Return "FINISH: {answer}"
    # Fill in below:
    pass
    
    return f"Unknown action: {action}"


        return f"FINISH: {match.group(1)}"print("✅ Tools defined!")

print("✅ Tools defined!")
    
    return f"Unknown action: {action}"

In [ ]:
# Define the ReAct prompt
# TODO: Complete the ReAct prompt by filling in the available actions
react_prompt = """Solve tasks by alternating Thought, Action, Observation steps.

Available actions:
- compare_numbers[a, b]: Compare two numbers and return which is larger
- add_numbers[a, b]: # TODO: Fill in the description
- get_current_date[]: # TODO: Fill in the description
- finish[answer]: # TODO: Fill in the description
"""

# Few-shot example to teach the model the format
cot_prompt = """
Example:
Question: What is 5 + 3, and is the result larger than 7?
Thought 1: I need to first add 5 and 3.
Action 1: add_numbers[5, 3]
Observation 1: 5 + 3 = 8
Thought 2: The sum is 8. Now I need to compare 8 with 7.
Action 2: compare_numbers[8, 7]
Observation 2: 8 > 7 is TRUE. 8 is greater than 7.
Thought 3: 8 is larger than 7, so I have my answer.
Action 3: finish[The sum is 8, which is larger than 7]
"""


def apply_system_prompt(question: str, include_cot=True) -> str:
    """Build the full prompt for the ReAct agent."""
    prompt = react_prompt
    if include_cot:
        prompt += cot_prompt
    prompt += f"\nQuestion: {question}\n"
    return prompt

print("✅ ReAct prompt ready!")

In [ ]:
# Implement the ReAct loop
def run_react_agent(question: str, max_steps: int = 7, verbose=False):
    """Run ReAct loop following the paper's format."""
    prompt = apply_system_prompt(question, include_cot=True)
    
    console.print(Panel(question, title="Question", border_style="blue"))
    
    for i in range(1, max_steps + 1):
        # Generate thought and action together
        thought_action = generate(prompt + f"Thought {i}:")
        
        if verbose:
            console.print(Panel(prompt + f"Thought {i}:", title=f"Raw Prompt: Step {i}", border_style="black"))
            console.print(Panel(thought_action, title=f"Raw Response: Step {i}", border_style="black"))
        
        # Parse thought and action from response
        lines = thought_action.strip().split("\n")
        thought = lines[0].strip() if len(lines) > 0 else ""
        action = lines[1].strip() if len(lines) > 1 else ""
        
        # Clean up action line
        action = re.sub(r'^Action\s*\d*:\s*', '', action)
        
        console.print(Panel(f"Thought {i}: {thought}", title=f"Step {i}", border_style="green"))
        console.print(Panel(f"Action {i}: {action}", border_style="cyan"))
        
        # Execute the action
        observation = execute_action(action)
        console.print(Panel(f"Observation {i}: {observation}", border_style="yellow"))
        
        # Check if finished
        if observation.startswith("FINISH:"):
            answer = observation[7:].strip()
            console.print(Panel(f"Final Answer: {answer}", title="Done", border_style="magenta"))
            return answer
        
        # TODO: Add the current step to the prompt for the next iteration
        # Hint: Create a string with Thought {i}, Action {i}, and Observation {i}
        # Then append it to the prompt variable
        # Fill in below:
        pass
    
    console.print("[red]Max steps reached without finishing[/red]")
    return None

print("✅ ReAct agent ready!")

In [ ]:
# Test the ReAct agent
question = "9.11 and 9.9, which number is larger?"

# Set verbose=True to see the full prompt and raw model outputs
run_react_agent(question, max_steps=7, verbose=False)

### 💡 Try It Yourself!

Experiment with different questions:
```python
# Arithmetic
run_react_agent("What is 9.11 + 9.9?", max_steps=7, verbose=False)

# Current information
run_react_agent("What day is today?", max_steps=7, verbose=False)

# Multi-step reasoning
run_react_agent("What is 5 + 3, and is the result larger than 7?", max_steps=7, verbose=False)
```

## Part 3: LangChain with Local Models

LangChain is a framework that simplifies building LLM applications with tools. Instead of manually implementing the ReAct loop, LangChain provides abstractions for:
- Defining tools with decorators
- Managing conversation history
- Parsing tool calls from model outputs

Let's see how to use LangChain with the same local model.

In [ ]:
# Set up LangChain with our local model
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_core.tools import tool

# Create a pipeline specifically for LangChain
# We reuse the same model and tokenizer from earlier
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    max_new_tokens=256,
    return_full_text=False,
)

# Wrap the pipeline for LangChain
llm = HuggingFacePipeline(pipeline=pipe)
chat_model_local = ChatHuggingFace(llm=llm)

# Define tools using LangChain's @tool decorator
@tool
def lc_compare_numbers(a: float, b: float) -> str:
    """Compare two numbers and return which is greater."""
    # TODO: Implement the comparison logic (same as before)
    # Fill in below:
    pass

@tool
def lc_add_numbers(a: float, b: float) -> str:
    """Add two numbers and return the sum."""
    # TODO: Implement addition (same as before)
    # Fill in below:
    pass

@tool
def lc_get_current_date() -> str:
    """Get the current date."""
    # TODO: Return today's date (same as before)
    # Fill in below:
    pass

tools_local = {
    "compare_numbers": lc_compare_numbers,
    "add_numbers": lc_add_numbers,
    "get_current_date": lc_get_current_date,
}

print("✅ LangChain local setup complete!")

In [ ]:
# System prompt for tool calling with LangChain
SYSTEM_PROMPT = """You have access to tools. To call a tool, output:
<tool_call>{"name": "tool_name", "arguments": {"arg1": value1}}</tool_call>

Available tools:
- compare_numbers(a, b): Compare two numbers and return which is larger
- add_numbers(a, b): Add two numbers together
- get_current_date(): Get the current date

When you have the final answer, just respond normally without a tool call.

Example:
User: What is 5 + 3, and is the result larger than 7?
Assistant: I'll add the numbers first.
<tool_call>{"name": "add_numbers", "arguments": {"a": 5, "b": 3}}</tool_call>

[Tool result: 5 + 3 is 8.]

Now I'll compare 8 with 7.
<tool_call>{"name": "compare_numbers", "arguments": {"a": 8, "b": 7}}</tool_call>

[Tool result: 8 > 7 is TRUE. 8 is greater than 7.]

The sum of 5 + 3 is 8, which is larger than 7.
"""


def parse_tool_call(text: str):
    """Extract tool call from <tool_call>...</tool_call> tags."""
    # TODO: Use regex to find the content between <tool_call> and </tool_call> tags
    # Hint: Use re.search() with the pattern r'<tool_call>(.*?)</tool_call>'
    # Then try to parse the JSON and return it, or return None if it fails
    # Fill in below:
    pass

print("✅ System prompt ready!")

In [ ]:
# Run the LangChain agent with local model
def run_langchain_local_agent(question: str, max_steps: int = 5, verbose: bool = False):
    """Run agent loop with LangChain and local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    
    console.print(Panel(question, title="Question", border_style="blue"))
    
    for step in range(max_steps):
        if verbose:
            console.print(Panel(str(messages), title=f"Raw Prompt: Step {step + 1}", border_style="black"))
        
        # Get model response
        response = chat_model_local.invoke(messages)
        response_text = response.content
        
        if verbose:
            console.print(Panel(response_text, title=f"Raw Response: Step {step + 1}", border_style="black"))
        
        # Check for tool call
        tool_call = parse_tool_call(response_text)
        
        if tool_call:
            tool_name = tool_call["name"]
            tool_args = tool_call.get("arguments", {})
            
            # Display action
            console.print(Panel(
                f'{tool_name}({tool_args})',
                title="Action",
                border_style="cyan"
            ))
            
            # Execute tool
            if tool_name in tools_local:
                result = tools_local[tool_name].invoke(tool_args)
                console.print(Panel(result, title="Observation", border_style="yellow"))
                
                # TODO: Add the assistant's response and tool result to the conversation
                # Hint: Append two messages to the messages list:
                #   1. {"role": "assistant", "content": response_text}
                #   2. {"role": "user", "content": f"[Tool result: {result}]"}
                # Fill in below:
                pass
            else:
                console.print(f"[red]Unknown tool: {tool_name}[/red]")
                break
        else:
            # No tool call - this is the final answer
            console.print(Panel(response_text, title="Answer", border_style="green"))
            return response_text
    
    console.print("[red]Max steps reached[/red]")

    return Noneprint("✅ LangChain local agent ready!")


In [ ]:
# Test the LangChain local agent
question = "9.11 and 9.9, which number is larger?"
run_langchain_local_agent(question, verbose=False)

## Part 4: LangChain with API Models

While local models are great for learning and experimentation, production AI agents often use API-based models like:
- **Anthropic Claude** - Excellent at following instructions and tool use
- **OpenAI GPT-4** - Strong reasoning capabilities
- **Google Gemini** - Fast and capable

These models have native tool-calling capabilities built in. 

**Note**: This section is already implemented and doesn't require fill-ins, but you can experiment with it if you have an API key.

In [ ]:
# Set up environment variables for API access
from dotenv import load_dotenv
import os

load_dotenv()

# Check if API key is available
if "ANTHROPIC_API_KEY" in os.environ:
    print("✅ Anthropic API key found")
else:
    print("⚠️  No Anthropic API key found. Set ANTHROPIC_API_KEY in your .env file")
    print("   You can skip this section if you don't have an API key.")

In [ ]:
# Set up Claude API model with LangChain
import sys

# Check Python version compatibility
python_version = sys.version_info
if python_version >= (3, 14):
    print(f"⚠️  Warning: Python {python_version.major}.{python_version.minor} detected")
    print("   LangChain may have compatibility issues with Python 3.14+")
    print("   Consider using Python 3.11 or 3.12 for best compatibility")
    print("\n   Skipping LangChain API agent setup due to compatibility issues.")
    print("   The API can still be used directly without LangChain.")
else:
    try:
        from langchain_anthropic import ChatAnthropic
        from langchain.agents import create_tool_calling_agent, AgentExecutor
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_core.callbacks import BaseCallbackHandler
        
        class ToolCallbackHandler(BaseCallbackHandler):
            """Callback handler to display tool calls with rich panels."""
            
            def on_tool_start(self, serialized, input_str, **kwargs):
                tool_name = serialized.get("name", "unknown")
                console.print(Panel(f"{tool_name}({input_str})", title="Action", border_style="cyan"))
            
            def on_tool_end(self, output, **kwargs):
                content = output.content if hasattr(output, 'content') else str(output)
                console.print(Panel(content, title="Observation", border_style="yellow"))
        
        # Initialize Claude
        if "ANTHROPIC_API_KEY" in os.environ:
            chat_model_api = ChatAnthropic(model="claude-sonnet-4-20250514")
            
            # Define tools using LangChain decorators
            @tool
            def api_compare_numbers(a: float, b: float) -> str:
                """Compare two numbers and return which is greater."""
                if a > b:
                    return f"{a} > {b} is TRUE. {a} is greater than {b}."
                elif a < b:
                    return f"{a} > {b} is FALSE. {a} is less than {b}."
                else:
                    return f"{a} and {b} are equal."
            
            @tool
            def api_add_numbers(a: float, b: float) -> str:
                """Add two numbers and return the sum."""
                return f"{a} + {b} is {a + b}."
            
            @tool
            def api_get_current_date() -> str:
                """Get the current date."""
                return f"Today's date is {datetime.now().strftime('%A, %B %d, %Y')}."
            
            tools_api = [api_compare_numbers, api_add_numbers, api_get_current_date]
            
            # Create the agent
            prompt = ChatPromptTemplate.from_messages([
                ("system", "You are a helpful assistant with access to tools."),
                ("human", "{input}"),
                ("placeholder", "{agent_scratchpad}"),
            ])
            
            agent = create_tool_calling_agent(chat_model_api, tools_api, prompt)
            agent_executor = AgentExecutor(agent=agent, tools=tools_api, verbose=False)
            
            print("✅ Claude API agent ready!")
        else:
            print("⚠️  Skipping API agent setup (no API key found)")
    
    except (ImportError, TypeError) as e:
        print(f"⚠️  Error setting up LangChain agent: {e}")
        print("   This may be due to Python version compatibility issues.")
        print("   Consider using Python 3.11 or 3.12 for best compatibility.")

In [ ]:
# Test the Claude API agent
if python_version >= (3, 14):
    print("⚠️  Skipping API agent test (Python 3.14+ compatibility issues)")
    print("   See the cell above for more information.")
elif "ANTHROPIC_API_KEY" in os.environ and 'agent_executor' in globals():
    question = "9.11 and 9.9, which number is larger?"
    
    console.print(Panel(question, title="Question", border_style="blue"))
    
    result = agent_executor.invoke(
        {"input": question},
        config={"callbacks": [ToolCallbackHandler()]}
    )
    
    # Extract the final answer
    final_message = result["output"]
    console.print(Panel(final_message, title="Answer", border_style="green"))
elif "ANTHROPIC_API_KEY" not in os.environ:
    print("⚠️  Skipping API agent test (no API key found)")
else:
    print("⚠️  Skipping API agent test (agent not initialized)")

### 🎯 Key Advantages of API Models

1. **Native Tool Support**: No need for custom prompt engineering
2. **Better Reliability**: More consistent tool calling behavior
3. **Higher Accuracy**: Better at multi-step reasoning
4. **Lower Latency**: Optimized infrastructure

**Trade-offs**: Cost per request, requires internet connection, data privacy considerations

## Part 5: Model Context Protocol (MCP)

**MCP** is a new standard for connecting AI assistants to external tools and data sources. Instead of building custom integrations for each tool, MCP provides:

- **Universal protocol** for tool discovery and execution
- **Server-based architecture** where tools run independently
- **Easy integration** with AI coding assistants (Claude Code, Cursor, etc.)

### MCP Architecture

```
AI Assistant (Client)
    ↕ (MCP Protocol)
MCP Server
    ↕
Tools/Resources
```

The MCP server exposes tools that any compatible AI assistant can discover and use.

In [ ]:
# Display the MCP server code
# The actual server is in mcp_server.py in this directory

import os

mcp_server_path = os.path.join(os.path.dirname(os.path.abspath(".")), 
                                "assignments/module2/tool_calling_in_agents/solution/mcp_server.py")

print("📄 MCP Server Location:", mcp_server_path)
print("\n" + "="*70)

# Read and display the MCP server code
if os.path.exists(mcp_server_path):
    with open(mcp_server_path, 'r') as f:
        print(f.read())
else:
    # Fallback to relative path
    relative_path = "mcp_server.py"
    if os.path.exists(relative_path):
        with open(relative_path, 'r') as f:
            print(f.read())
    else:
        print("⚠️ mcp_server.py not found. Please check the file location.")
        
print("="*70)
print("\n✅ The MCP server is defined in mcp_server.py")
print("See the cells below for VS Code installation instructions.")

### Installing MCP Server in VS Code

To use the MCP server with VS Code and GitHub Copilot:

#### Step 1: Install Required Packages
```bash
pip install mcp fastmcp
```

#### Step 2: Configure VS Code Settings

Add the MCP server to your VS Code settings. Open your `settings.json` (Command Palette → ">Preferences: Open User Settings (JSON)") and add:

```json
{
  "github.copilot.chat.mcp.servers": {
    "tool-calling-demo": {
      "command": "python",
      "args": [
        "/absolute/path/to/CMU-AGT/assignments/module2/tool_calling_in_agents/solution/mcp_server.py"
      ]
    }
  }
}
```

**Important**: Replace `/absolute/path/to/` with the actual absolute path to your project directory.

#### Step 3: Restart VS Code

After adding the configuration, restart VS Code for the changes to take effect.

#### Step 4: Verify Installation

In GitHub Copilot Chat, ask it to use the tools:
- "Use compare_numbers to check if 9.11 is larger than 9.9"
- "What's the current date using get_current_date?"
- "Add 5 and 3 using add_numbers"

Copilot will automatically discover and use the MCP tools!

---

### Alternative: Using with Other AI Assistants

**For Claude Code:**
```bash
claude mcp add-json tool-calling-demo \
  '{"command": "python", "args": ["/path/to/mcp_server.py"]}'
```

**For Cursor:**
Configure in Cursor Settings → MCP Servers, similar to VS Code.

**For Gemini CLI:**
```bash
gemini mcp add tool-calling-demo python /path/to/mcp_server.py
```

### Testing the MCP Server Locally

You can also test the MCP server directly before integrating it with VS Code:

In [ ]:
# Test the MCP tools directly (without running the server)
# Import the tools from our MCP server file

import sys
import os

# Add the current directory to path to import mcp_server
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

try:
    # Import the tools from mcp_server
    from mcp_server import compare_numbers, add_numbers, multiply_numbers, get_current_date
    
    print("✅ Successfully imported MCP tools!\n")
    
    # Test each tool
    console.print(Panel("Testing MCP Tools", style="bold blue"))
    
    # Test compare_numbers
    result = compare_numbers(9.11, 9.9)
    console.print(Panel(f"compare_numbers(9.11, 9.9)\n→ {result}", border_style="green"))
    
    # Test add_numbers
    result = add_numbers(5, 3)
    console.print(Panel(f"add_numbers(5, 3)\n→ {result}", border_style="green"))
    
    # Test multiply_numbers
    result = multiply_numbers(4, 7)
    console.print(Panel(f"multiply_numbers(4, 7)\n→ {result}", border_style="green"))
    
    # Test get_current_date
    result = get_current_date()
    console.print(Panel(f"get_current_date()\n→ {result}", border_style="green"))
    
except ImportError as e:
    print(f"⚠️ Could not import from mcp_server.py: {e}")
    print("Make sure mcp_server.py is in the same directory as this notebook.")

### 📋 Getting the Absolute Path for VS Code Configuration

Use this cell to get the exact path you need for your VS Code settings:

In [ ]:
# Get the absolute path to mcp_server.py for VS Code configuration
import os

# Try multiple methods to find the file
current_dir = os.getcwd()
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else current_dir

# Construct the path
mcp_file = "mcp_server.py"
possible_paths = [
    os.path.join(current_dir, mcp_file),
    os.path.join(notebook_dir, mcp_file),
    os.path.abspath(mcp_file)
]

# Find which path exists
mcp_path = None
for path in possible_paths:
    if os.path.exists(path):
        mcp_path = os.path.abspath(path)
        break

if mcp_path:
    print("✅ Found mcp_server.py!")
    print("\n📋 Copy this path for your VS Code settings.json:\n")
    print("=" * 70)
    print(mcp_path)
    print("=" * 70)
    
    print("\n📝 Add this to your VS Code settings.json:")
    print("\n{")
    print('  "github.copilot.chat.mcp.servers": {')
    print('    "tool-calling-demo": {')
    print('      "command": "python",')
    print('      "args": [')
    print(f'        "{mcp_path}"')
    print('      ]')
    print('    }')
    print('  }')
    print("}")
else:
    print("⚠️ mcp_server.py not found in current directory")
    print(f"Current directory: {current_dir}")
    print("\nMake sure mcp_server.py is in the same directory as this notebook.")

## Summary and Comparison

Let's compare the different approaches:

| Approach | Pros | Cons | Best For |
|----------|------|------|----------|
| **No Tools** | Simple, no setup | Inaccurate, limited knowledge | Basic text generation |
| **ReAct** | Full control, transparent | Manual implementation, fragile | Learning, research |
| **LangChain (Local)** | Framework abstractions, offline | Smaller models less reliable | Development, testing |
| **LangChain (API)** | Reliable, accurate, easy | Costs money, internet required | Production |
| **MCP** | Standardized, reusable, universal | Newer ecosystem, setup needed | Tool integration |

### Key Takeaways

1. **Tools are essential** for tasks requiring computation, current info, or external data
2. **ReAct prompting** is a foundational technique but requires careful engineering
3. **Frameworks like LangChain** simplify development but add abstractions
4. **API models** provide the best reliability for production use cases
5. **MCP** is emerging as a standard for tool integration across AI assistants

## 🚀 Exercises

Try these challenges to deepen your understanding:

1. **Add a new tool**: Implement a `multiply_numbers` or `divide_numbers` tool and integrate it into the ReAct and LangChain agents

2. **Complex queries**: Test multi-step reasoning like "What is (9.11 + 9.9) compared to 20?"

3. **Error handling**: What happens when tools receive invalid inputs? Add validation

4. **Custom MCP server**: Create an MCP server with domain-specific tools (e.g., weather data, file operations)

5. **Compare model performance**: Try different base models (Phi-3, Llama, etc.) and compare their tool-calling abilities

6. **Optimize prompts**: Experiment with different ReAct prompt formulations for better performance

## 📚 Additional Resources

- **ReAct Paper**: [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)
- **LangChain Docs**: [Tool Calling Guide](https://python.langchain.com/docs/modules/agents/)
- **MCP Documentation**: [Model Context Protocol](https://modelcontextprotocol.io/)
- **Anthropic Tool Use**: [Claude Tool Use Guide](https://docs.anthropic.com/claude/docs/tool-use)